# Inżynieria cech
 - połączenie danych drużynowych z podsumowaniami meczów
 - przygotowanie cech na formę optymalną do użycia w modelach - normalizacja

In [7]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

pd.set_option('display.max_columns', 40)

In [8]:
con = sqlite3.connect('../data/transformed/team_moving_avgs.sqlite')
team_last_20 = pd.read_sql_query(f"SELECT * FROM \"{'team_last_20'}\"", con)
team_last_30 = pd.read_sql_query(f"SELECT * FROM \"{'team_last_30'}\"", con)
team_last_40 = pd.read_sql_query(f"SELECT * FROM \"{'team_last_40'}\"", con)
team_all_season = pd.read_sql_query(f"SELECT * FROM \"{'team_all_season'}\"", con)
con.close()

con = sqlite3.connect('../data/transformed/games.sqlite')
games = pd.read_sql_query(f"SELECT * FROM \"{'games'}\"", con)
con.close()

In [9]:
team_last_20

,id,game_id,Date,Season,team,win,streak,last10,FG,FGA,FG%,3P,3PA,3P%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,TS%,eFG%,3PAr,FTr,ORB%,DRB%,TRB%,AST%,STL%,BLK%,TOV%,ORtg,DRtg,Pace
0,0,171810170001,2017-10-17,1718,BOS,0,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
1,1,171810170001,2017-10-17,1718,CLE,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
2,2,171810170002,2017-10-17,1718,HOU,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
3,3,171810170002,2017-10-17,1718,GSW,0,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
4,4,171810180003,2017-10-18,1718,CHO,0,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16577,16577,232404141229,2024-04-14,2324,OKC,1,4,6.0,43.450000,88.100000,0.494350,12.90000,34.90,0.368350,17.050000,21.250000,0.814000,8.950000,33.700000,42.650000,27.350000,9.100000,5.950000,13.250000,17.450000,116.850000,0.600750,0.567250,0.395000,0.246250,21.235000,78.020000,50.005000,62.910000,9.090000,12.235000,11.92500,117.170000,112.425000,99.305000
16578,16578,232404141230,2024-04-14,2324,POR,0,-4,2.0,39.500000,89.950000,0.440150,11.00000,32.35,0.337250,13.200000,16.950000,0.792400,13.450000,31.900000,45.350000,24.050000,6.550000,3.900000,15.600000,18.550000,103.200000,0.530700,0.501450,0.360500,0.188400,28.620000,75.595000,51.005000,60.535000,6.780000,7.190000,13.84500,106.915000,117.750000,96.020000
16579,16579,232404141230,2024-04-14,2324,SAC,1,-3,3.0,40.850000,90.350000,0.452900,14.00000,39.65,0.352450,15.500000,19.650000,0.780250,11.550000,34.000000,45.550000,27.100000,8.500000,4.550000,12.300000,19.450000,111.200000,0.562700,0.530400,0.439050,0.218600,25.500000,79.125000,51.430000,66.555000,8.800000,8.875000,10.98500,114.995000,110.780000,96.260000
16580,16580,232404141231,2024-04-14,2324,DET,0,1,2.0,38.300000,85.800000,0.446500,10.55000,32.50,0.323600,16.800000,20.700000,0.797950,9.300000,34.450000,43.750000,23.150000,7.400000,4.100000,15.150000,17.800000,103.950000,0.548300,0.508250,0.380700,0.243700,21.410000,78.715000,50.075000,60.630000,7.490000,7.840000,13.68500,105.235000,114.295000,98.760000


In [10]:
games

,index,TEAM_NAME,GP,W,L,W_PCT,GP_RANK,W_PCT_RANK,PLUS_MINUS_RANK,Date,TEAM_NAME.1,GP.1,W.1,L.1,W_PCT.1,GP_RANK.1,W_PCT_RANK.1,PLUS_MINUS_RANK.1,Days-Rest-Home,Days-Rest-Away
0,0,Cleveland Cavaliers,1.0,1.0,0.0,1.000,1.0,1.0,1.0,2017-10-17,Boston Celtics,1.0,0.0,1.0,0.000,1.0,3.0,4.0,10.0,10.0
1,1,Golden State Warriors,1.0,0.0,1.0,0.000,1.0,3.0,3.0,2017-10-17,Houston Rockets,1.0,1.0,0.0,1.000,1.0,1.0,2.0,10.0,10.0
2,2,Indiana Pacers,1.0,1.0,0.0,1.000,3.0,1.0,5.0,2017-10-18,Brooklyn Nets,1.0,0.0,1.0,0.000,3.0,13.0,20.0,10.0,10.0
3,3,Washington Wizards,1.0,1.0,0.0,1.000,3.0,1.0,10.0,2017-10-18,Philadelphia 76ers,1.0,0.0,1.0,0.000,3.0,13.0,14.0,10.0,10.0
4,4,Orlando Magic,1.0,1.0,0.0,1.000,3.0,1.0,8.0,2017-10-18,Miami Heat,1.0,0.0,1.0,0.000,3.0,13.0,18.0,10.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8770,8770,Los Angeles Lakers,82.0,47.0,35.0,0.573,1.0,12.0,19.0,2024-04-27,Denver Nuggets,82.0,57.0,25.0,0.695,1.0,2.0,4.0,2.0,2.0
8771,8771,Philadelphia 76ers,82.0,47.0,35.0,0.573,1.0,12.0,9.0,2024-04-28,New York Knicks,82.0,50.0,32.0,0.610,1.0,6.0,5.0,3.0,3.0
8772,8772,Dallas Mavericks,82.0,50.0,32.0,0.610,1.0,6.0,14.0,2024-04-28,LA Clippers,82.0,51.0,31.0,0.622,1.0,5.0,7.0,2.0,2.0
8773,8773,Indiana Pacers,82.0,47.0,35.0,0.573,1.0,12.0,9.0,2024-04-28,Milwaukee Bucks,82.0,49.0,33.0,0.598,1.0,8.0,11.0,2.0,2.0


In [11]:
games_minus1 = games.copy()
games_minus1['Date'] = (pd.to_datetime(games_minus1['Date']) - pd.Timedelta(days=1)).dt.strftime('%Y-%m-%d')

games_plus1 = games.copy()
games_plus1['Date'] = (pd.to_datetime(games_plus1['Date']) + pd.Timedelta(days=1)).dt.strftime('%Y-%m-%d')

# games_minus2 = games.copy()
# games_minus1['Date'] = (pd.to_datetime(games_minus1['Date']) - pd.Timedelta(days=2)).dt.strftime('%Y-%m-%d')
#
# games_plus2 = games.copy()
# games_plus1['Date'] = (pd.to_datetime(games_plus1['Date']) + pd.Timedelta(days=2)).dt.strftime('%Y-%m-%d')


games_original = games.copy()
games_original['Date'] = pd.to_datetime(games_original['Date']).dt.strftime('%Y-%m-%d')

# Łączenie wszystkich wersji tabeli 'games' w jeden DataFrame
games_expanded = pd.concat([games_original, games_minus1, games_plus1])

games = pd.concat([games, games_minus1, games_plus1])

###

In [12]:
away_df = team_last_20[team_last_20.index % 2 == 0].reset_index(drop=True)
home_df = team_last_20[team_last_20.index % 2 == 1].reset_index(drop=True)

away_df = away_df.add_prefix('away_')
team_last_20 = pd.concat([home_df, away_df], axis=1) \
            .drop(columns=['away_id', 'id', 'away_game_id', 'away_Season', 'away_win', 'away_Date']) \
            .rename(columns={col: f'home_{col}' for col in team_last_20.columns[4:40]})
team_last_20

,game_id,Date,Season,home_team,home_win,home_streak,home_last10,home_FG,home_FGA,home_FG%,home_3P,home_3PA,home_3P%,home_FT,home_FTA,home_FT%,home_ORB,home_DRB,home_TRB,home_AST,...,away_AST,away_STL,away_BLK,away_TOV,away_PF,away_PTS,away_TS%,away_eFG%,away_3PAr,away_FTr,away_ORB%,away_DRB%,away_TRB%,away_AST%,away_STL%,away_BLK%,away_TOV%,away_ORtg,away_DRtg,away_Pace
0,171810170001,2017-10-17,1718,CLE,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
1,171810170002,2017-10-17,1718,GSW,0,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
2,171810180003,2017-10-18,1718,DET,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
3,171810180004,2017-10-18,1718,IND,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
4,171810180005,2017-10-18,1718,ORL,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8286,232404141227,2024-04-14,2324,MIN,0,1,7.0,41.900000,87.300000,0.480750,13.20000,34.60,0.378800,15.500000,20.850000,0.744600,9.700000,32.450000,42.150000,27.650000,...,28.650000,7.200000,5.750000,14.650000,16.400000,113.200000,0.600900,0.571200,0.406200,0.206550,24.390000,78.980000,52.855000,67.600000,7.355000,10.820000,13.45500,116.540000,113.095000,96.705000
8287,232404141228,2024-04-14,2324,NOP,0,4,5.0,41.250000,85.450000,0.483200,13.00000,33.20,0.382050,16.350000,20.500000,0.800050,8.750000,34.100000,42.850000,27.050000,...,28.500000,6.100000,5.300000,14.750000,14.050000,120.350000,0.618500,0.582550,0.387450,0.292650,20.590000,78.070000,50.890000,65.405000,6.005000,9.310000,13.12500,118.005000,115.450000,100.930000
8288,232404141229,2024-04-14,2324,OKC,1,4,6.0,43.450000,88.100000,0.494350,12.90000,34.90,0.368350,17.050000,21.250000,0.814000,8.950000,33.700000,42.650000,27.350000,...,27.300000,7.250000,6.300000,12.700000,17.400000,116.850000,0.596850,0.569850,0.427800,0.236850,21.355000,79.130000,51.300000,62.640000,7.305000,11.480000,11.42500,117.455000,108.015000,98.895000
8289,232404141230,2024-04-14,2324,SAC,1,-3,3.0,40.850000,90.350000,0.452900,14.00000,39.65,0.352450,15.500000,19.650000,0.780250,11.550000,34.000000,45.550000,27.100000,...,24.050000,6.550000,3.900000,15.600000,18.550000,103.200000,0.530700,0.501450,0.360500,0.188400,28.620000,75.595000,51.005000,60.535000,6.780000,7.190000,13.84500,106.915000,117.750000,96.020000


In [13]:
away_df = team_last_30[team_last_30.index % 2 == 0].reset_index(drop=True)
home_df = team_last_30[team_last_30.index % 2 == 1].reset_index(drop=True)
away_df = away_df.add_prefix('away_')
team_last_30 = pd.concat([home_df, away_df], axis=1) \
            .drop(columns=['away_id', 'id', 'away_game_id', 'away_Season', 'away_win', 'away_Date']) \
            .rename(columns={col: f'home_{col}' for col in team_last_30.columns[4:40]})


away_df = team_last_40[team_last_40.index % 2 == 0].reset_index(drop=True)
home_df = team_last_40[team_last_40.index % 2 == 1].reset_index(drop=True)
away_df = away_df.add_prefix('away_')
team_last_40 = pd.concat([home_df, away_df], axis=1) \
            .drop(columns=['away_id', 'id', 'away_game_id', 'away_Season', 'away_win', 'away_Date']) \
            .rename(columns={col: f'home_{col}' for col in team_last_40.columns[4:40]})


away_df = team_all_season[team_all_season.index % 2 == 0].reset_index(drop=True)
home_df = team_all_season[team_all_season.index % 2 == 1].reset_index(drop=True)
away_df = away_df.add_prefix('away_')
team_all_season = pd.concat([home_df, away_df], axis=1) \
            .drop(columns=['away_id', 'id', 'away_game_id', 'away_Season', 'away_win', 'away_Date']) \
            .rename(columns={col: f'home_{col}' for col in team_all_season.columns[4:40]})

In [14]:
team_last_20

,game_id,Date,Season,home_team,home_win,home_streak,home_last10,home_FG,home_FGA,home_FG%,home_3P,home_3PA,home_3P%,home_FT,home_FTA,home_FT%,home_ORB,home_DRB,home_TRB,home_AST,...,away_AST,away_STL,away_BLK,away_TOV,away_PF,away_PTS,away_TS%,away_eFG%,away_3PAr,away_FTr,away_ORB%,away_DRB%,away_TRB%,away_AST%,away_STL%,away_BLK%,away_TOV%,away_ORtg,away_DRtg,away_Pace
0,171810170001,2017-10-17,1718,CLE,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
1,171810170002,2017-10-17,1718,GSW,0,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
2,171810180003,2017-10-18,1718,DET,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
3,171810180004,2017-10-18,1718,IND,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
4,171810180005,2017-10-18,1718,ORL,1,0,0.0,39.607724,86.060569,0.461134,10.49065,29.00,0.361003,16.627236,21.676829,0.768602,9.711789,33.805285,43.517073,23.236992,...,23.236992,7.716667,4.815447,14.262602,19.852439,106.333333,0.557316,0.522378,0.337896,0.254894,22.140163,77.860894,50.000447,58.535935,7.873902,8.448008,12.98252,108.668699,108.668699,97.318537
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8286,232404141227,2024-04-14,2324,MIN,0,1,7.0,41.900000,87.300000,0.480750,13.20000,34.60,0.378800,15.500000,20.850000,0.744600,9.700000,32.450000,42.150000,27.650000,...,28.650000,7.200000,5.750000,14.650000,16.400000,113.200000,0.600900,0.571200,0.406200,0.206550,24.390000,78.980000,52.855000,67.600000,7.355000,10.820000,13.45500,116.540000,113.095000,96.705000
8287,232404141228,2024-04-14,2324,NOP,0,4,5.0,41.250000,85.450000,0.483200,13.00000,33.20,0.382050,16.350000,20.500000,0.800050,8.750000,34.100000,42.850000,27.050000,...,28.500000,6.100000,5.300000,14.750000,14.050000,120.350000,0.618500,0.582550,0.387450,0.292650,20.590000,78.070000,50.890000,65.405000,6.005000,9.310000,13.12500,118.005000,115.450000,100.930000
8288,232404141229,2024-04-14,2324,OKC,1,4,6.0,43.450000,88.100000,0.494350,12.90000,34.90,0.368350,17.050000,21.250000,0.814000,8.950000,33.700000,42.650000,27.350000,...,27.300000,7.250000,6.300000,12.700000,17.400000,116.850000,0.596850,0.569850,0.427800,0.236850,21.355000,79.130000,51.300000,62.640000,7.305000,11.480000,11.42500,117.455000,108.015000,98.895000
8289,232404141230,2024-04-14,2324,SAC,1,-3,3.0,40.850000,90.350000,0.452900,14.00000,39.65,0.352450,15.500000,19.650000,0.780250,11.550000,34.000000,45.550000,27.100000,...,24.050000,6.550000,3.900000,15.600000,18.550000,103.200000,0.530700,0.501450,0.360500,0.188400,28.620000,75.595000,51.005000,60.535000,6.780000,7.190000,13.84500,106.915000,117.750000,96.020000


In [15]:
nba_teams_mapping = {
    'ATL': 'Atlanta Hawks',
    'BOS': 'Boston Celtics',
    'BRK': 'Brooklyn Nets',
    'CHO': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls',
    'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks',
    'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons',
    'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets',
    'IND': 'Indiana Pacers',
    'LAC': 'LA Clippers',
    'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies',
    'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks',
    'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans',
    'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder',
    'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers',
    'PHO': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers',
    'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs',
    'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz',
    'WAS': 'Washington Wizards'
}

In [16]:
team_last_20['home_team_full'] = team_last_20['home_team'].map(nba_teams_mapping)
team_last_20['away_team_full'] = team_last_20['away_team'].map(nba_teams_mapping)

In [17]:
team_avgs_last_20 = pd.merge(
    team_last_20,
    games,
    how='left',
    left_on=['Date', 'home_team_full', 'away_team_full'],
    right_on=['Date', 'TEAM_NAME', 'TEAM_NAME.1']) \
    .drop(columns=['GP.1', 'TEAM_NAME.1', 'W.1', 'L.1', 'GP', 'TEAM_NAME',
                   'W', 'L', 'index', 'home_team_full', 'away_team_full'])

In [18]:
team_avgs_last_20
team_avgs_last_20.columns

Index(['game_id', 'Date', 'Season', 'home_team', 'home_win', 'home_streak',
       'home_last10', 'home_FG', 'home_FGA', 'home_FG%', 'home_3P', 'home_3PA',
       'home_3P%', 'home_FT', 'home_FTA', 'home_FT%', 'home_ORB', 'home_DRB',
       'home_TRB', 'home_AST', 'home_STL', 'home_BLK', 'home_TOV', 'home_PF',
       'home_PTS', 'home_TS%', 'home_eFG%', 'home_3PAr', 'home_FTr',
       'home_ORB%', 'home_DRB%', 'home_TRB%', 'home_AST%', 'home_STL%',
       'home_BLK%', 'home_TOV%', 'home_ORtg', 'home_DRtg', 'home_Pace',
       'away_team', 'away_streak', 'away_last10', 'away_FG', 'away_FGA',
       'away_FG%', 'away_3P', 'away_3PA', 'away_3P%', 'away_FT', 'away_FTA',
       'away_FT%', 'away_ORB', 'away_DRB', 'away_TRB', 'away_AST', 'away_STL',
       'away_BLK', 'away_TOV', 'away_PF', 'away_PTS', 'away_TS%', 'away_eFG%',
       'away_3PAr', 'away_FTr', 'away_ORB%', 'away_DRB%', 'away_TRB%',
       'away_AST%', 'away_STL%', 'away_BLK%', 'away_TOV%', 'away_ORtg',
       'away_DRtg', 'awa

In [19]:
team_last_30['home_team_full'] = team_last_30['home_team'].map(nba_teams_mapping)
team_last_30['away_team_full'] = team_last_30['away_team'].map(nba_teams_mapping)
team_avgs_last_30 = pd.merge(
    team_last_30,
    games,
    how='left',
    left_on=['Date', 'home_team_full', 'away_team_full'],
    right_on=['Date', 'TEAM_NAME', 'TEAM_NAME.1']) \
    .drop(columns=['GP.1', 'TEAM_NAME.1', 'W.1', 'L.1', 'GP', 'TEAM_NAME',
                   'W', 'L', 'index', 'home_team_full', 'away_team_full'])

team_last_40['home_team_full'] = team_last_40['home_team'].map(nba_teams_mapping)
team_last_40['away_team_full'] = team_last_40['away_team'].map(nba_teams_mapping)
team_avgs_last_40 = pd.merge(
    team_last_40,
    games,
    how='left',
    left_on=['Date', 'home_team_full', 'away_team_full'],
    right_on=['Date', 'TEAM_NAME', 'TEAM_NAME.1']) \
    .drop(columns=['GP.1', 'TEAM_NAME.1', 'W.1', 'L.1', 'GP', 'TEAM_NAME',
                   'W', 'L', 'index', 'home_team_full', 'away_team_full'])


team_all_season['home_team_full'] = team_all_season['home_team'].map(nba_teams_mapping)
team_all_season['away_team_full'] = team_all_season['away_team'].map(nba_teams_mapping)
team_avgs_all_season = pd.merge(
    team_all_season,
    games,
    how='left',
    left_on=['Date', 'home_team_full', 'away_team_full'],
    right_on=['Date', 'TEAM_NAME', 'TEAM_NAME.1']) \
    .drop(columns=['GP.1', 'TEAM_NAME.1', 'W.1', 'L.1', 'GP', 'TEAM_NAME',
                   'W', 'L', 'index', 'home_team_full', 'away_team_full'])

Użycie .rename w powyższym łańcuchu operacji nie zmieniało nazw kolumn, stąd przypisanie nazw jak poniżej

In [20]:
team_avgs_all_season.columns = [
    'game_id', 'Date', 'Season', 'home_team', 'home_win', 'home_streak',
    'home_last10', 'home_FG', 'home_FGA', 'home_FG%', 'home_3P', 'home_3PA',
    'home_3P%', 'home_FT', 'home_FTA', 'home_FT%', 'home_ORB', 'home_DRB',
    'home_TRB', 'home_AST', 'home_STL', 'home_BLK', 'home_TOV', 'home_PF',
    'home_PTS', 'home_TS%', 'home_eFG%', 'home_3PAr', 'home_FTr',
    'home_ORB%', 'home_DRB%', 'home_TRB%', 'home_AST%', 'home_STL%',
    'home_BLK%', 'home_TOV%', 'home_ORtg', 'home_DRtg', 'home_Pace',
    'away_team', 'away_streak', 'away_last10', 'away_FG', 'away_FGA',
    'away_FG%', 'away_3P', 'away_3PA', 'away_3P%', 'away_FT', 'away_FTA',
    'away_FT%', 'away_ORB', 'away_DRB', 'away_TRB', 'away_AST', 'away_STL',
    'away_BLK', 'away_TOV', 'away_PF', 'away_PTS', 'away_TS%', 'away_eFG%',
    'away_3PAr', 'away_FTr', 'away_ORB%', 'away_DRB%', 'away_TRB%',
    'away_AST%', 'away_STL%', 'away_BLK%', 'away_TOV%', 'away_ORtg',
    'away_DRtg', 'away_Pace', 'home_W_pct', 'home_GP_rank', 'home_W_pct_rank',
    'home_+/-_rank', 'away_W_pct', 'away_GP_rank', 'away_W_pct_rank',
    'away_+/-_rank', 'home_days_rest', 'away_days_rest'
]

team_avgs_last_40.columns = [
    'game_id', 'Date', 'Season', 'home_team', 'home_win', 'home_streak',
    'home_last10', 'home_FG', 'home_FGA', 'home_FG%', 'home_3P', 'home_3PA',
    'home_3P%', 'home_FT', 'home_FTA', 'home_FT%', 'home_ORB', 'home_DRB',
    'home_TRB', 'home_AST', 'home_STL', 'home_BLK', 'home_TOV', 'home_PF',
    'home_PTS', 'home_TS%', 'home_eFG%', 'home_3PAr', 'home_FTr',
    'home_ORB%', 'home_DRB%', 'home_TRB%', 'home_AST%', 'home_STL%',
    'home_BLK%', 'home_TOV%', 'home_ORtg', 'home_DRtg', 'home_Pace',
    'away_team', 'away_streak', 'away_last10', 'away_FG', 'away_FGA',
    'away_FG%', 'away_3P', 'away_3PA', 'away_3P%', 'away_FT', 'away_FTA',
    'away_FT%', 'away_ORB', 'away_DRB', 'away_TRB', 'away_AST', 'away_STL',
    'away_BLK', 'away_TOV', 'away_PF', 'away_PTS', 'away_TS%', 'away_eFG%',
    'away_3PAr', 'away_FTr', 'away_ORB%', 'away_DRB%', 'away_TRB%',
    'away_AST%', 'away_STL%', 'away_BLK%', 'away_TOV%', 'away_ORtg',
    'away_DRtg', 'away_Pace', 'home_W_pct', 'home_GP_rank', 'home_W_pct_rank',
    'home_+/-_rank', 'away_W_pct', 'away_GP_rank', 'away_W_pct_rank',
    'away_+/-_rank', 'home_days_rest', 'away_days_rest'
]

team_avgs_last_30.columns = [
    'game_id', 'Date', 'Season', 'home_team', 'home_win', 'home_streak',
    'home_last10', 'home_FG', 'home_FGA', 'home_FG%', 'home_3P', 'home_3PA',
    'home_3P%', 'home_FT', 'home_FTA', 'home_FT%', 'home_ORB', 'home_DRB',
    'home_TRB', 'home_AST', 'home_STL', 'home_BLK', 'home_TOV', 'home_PF',
    'home_PTS', 'home_TS%', 'home_eFG%', 'home_3PAr', 'home_FTr',
    'home_ORB%', 'home_DRB%', 'home_TRB%', 'home_AST%', 'home_STL%',
    'home_BLK%', 'home_TOV%', 'home_ORtg', 'home_DRtg', 'home_Pace',
    'away_team', 'away_streak', 'away_last10', 'away_FG', 'away_FGA',
    'away_FG%', 'away_3P', 'away_3PA', 'away_3P%', 'away_FT', 'away_FTA',
    'away_FT%', 'away_ORB', 'away_DRB', 'away_TRB', 'away_AST', 'away_STL',
    'away_BLK', 'away_TOV', 'away_PF', 'away_PTS', 'away_TS%', 'away_eFG%',
    'away_3PAr', 'away_FTr', 'away_ORB%', 'away_DRB%', 'away_TRB%',
    'away_AST%', 'away_STL%', 'away_BLK%', 'away_TOV%', 'away_ORtg',
    'away_DRtg', 'away_Pace', 'home_W_pct', 'home_GP_rank', 'home_W_pct_rank',
    'home_+/-_rank', 'away_W_pct', 'away_GP_rank', 'away_W_pct_rank',
    'away_+/-_rank', 'home_days_rest', 'away_days_rest'
]

team_avgs_last_20.columns = [
    'game_id', 'Date', 'Season', 'home_team', 'home_win', 'home_streak',
    'home_last10', 'home_FG', 'home_FGA', 'home_FG%', 'home_3P', 'home_3PA',
    'home_3P%', 'home_FT', 'home_FTA', 'home_FT%', 'home_ORB', 'home_DRB',
    'home_TRB', 'home_AST', 'home_STL', 'home_BLK', 'home_TOV', 'home_PF',
    'home_PTS', 'home_TS%', 'home_eFG%', 'home_3PAr', 'home_FTr',
    'home_ORB%', 'home_DRB%', 'home_TRB%', 'home_AST%', 'home_STL%',
    'home_BLK%', 'home_TOV%', 'home_ORtg', 'home_DRtg', 'home_Pace',
    'away_team', 'away_streak', 'away_last10', 'away_FG', 'away_FGA',
    'away_FG%', 'away_3P', 'away_3PA', 'away_3P%', 'away_FT', 'away_FTA',
    'away_FT%', 'away_ORB', 'away_DRB', 'away_TRB', 'away_AST', 'away_STL',
    'away_BLK', 'away_TOV', 'away_PF', 'away_PTS', 'away_TS%', 'away_eFG%',
    'away_3PAr', 'away_FTr', 'away_ORB%', 'away_DRB%', 'away_TRB%',
    'away_AST%', 'away_STL%', 'away_BLK%', 'away_TOV%', 'away_ORtg',
    'away_DRtg', 'away_Pace', 'home_W_pct', 'home_GP_rank', 'home_W_pct_rank',
    'home_+/-_rank', 'away_W_pct', 'away_GP_rank', 'away_W_pct_rank',
    'away_+/-_rank', 'home_days_rest', 'away_days_rest'
]


In [21]:
team_avgs_all_season.columns

Index(['game_id', 'Date', 'Season', 'home_team', 'home_win', 'home_streak',
       'home_last10', 'home_FG', 'home_FGA', 'home_FG%', 'home_3P', 'home_3PA',
       'home_3P%', 'home_FT', 'home_FTA', 'home_FT%', 'home_ORB', 'home_DRB',
       'home_TRB', 'home_AST', 'home_STL', 'home_BLK', 'home_TOV', 'home_PF',
       'home_PTS', 'home_TS%', 'home_eFG%', 'home_3PAr', 'home_FTr',
       'home_ORB%', 'home_DRB%', 'home_TRB%', 'home_AST%', 'home_STL%',
       'home_BLK%', 'home_TOV%', 'home_ORtg', 'home_DRtg', 'home_Pace',
       'away_team', 'away_streak', 'away_last10', 'away_FG', 'away_FGA',
       'away_FG%', 'away_3P', 'away_3PA', 'away_3P%', 'away_FT', 'away_FTA',
       'away_FT%', 'away_ORB', 'away_DRB', 'away_TRB', 'away_AST', 'away_STL',
       'away_BLK', 'away_TOV', 'away_PF', 'away_PTS', 'away_TS%', 'away_eFG%',
       'away_3PAr', 'away_FTr', 'away_ORB%', 'away_DRB%', 'away_TRB%',
       'away_AST%', 'away_STL%', 'away_BLK%', 'away_TOV%', 'away_ORtg',
       'away_DRtg', 'awa

In [22]:
def missing_data_summary(data):
    return data[data.isnull().any(axis=1)], \
        (data.isnull().mean() * 100).round(3)

rows_with_nan, missing_percentages = missing_data_summary(team_avgs_last_30)
rows_with_nan

,game_id,Date,Season,home_team,home_win,home_streak,home_last10,home_FG,home_FGA,home_FG%,home_3P,home_3PA,home_3P%,home_FT,home_FTA,home_FT%,home_ORB,home_DRB,home_TRB,home_AST,...,away_ORB%,away_DRB%,away_TRB%,away_AST%,away_STL%,away_BLK%,away_TOV%,away_ORtg,away_DRtg,away_Pace,home_W_pct,home_GP_rank,home_W_pct_rank,home_+/-_rank,away_W_pct,away_GP_rank,away_W_pct_rank,away_+/-_rank,home_days_rest,away_days_rest
5880,222310180001,2022-10-18,2223,BOS,1,0,0.0,42.933333,86.633333,0.496867,14.600000,38.166667,0.379633,17.266667,21.466667,0.810767,9.966667,35.433333,45.400000,27.566667,...,19.936667,78.843333,50.076667,63.683333,7.813333,8.610000,11.703333,116.563333,113.900000,96.706667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5881,222310180002,2022-10-18,2223,GSW,1,0,0.0,40.733333,86.766667,0.469233,14.166667,38.766667,0.365967,15.833333,20.266667,0.786933,9.433333,34.300000,43.733333,26.300000,...,21.270000,75.660000,48.386667,56.883333,6.883333,8.156667,12.583333,112.343333,118.013333,99.566667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7940,232402150819,2024-02-15,2324,MEM,1,1,1.0,38.733333,86.733333,0.446767,14.300000,39.600000,0.362567,16.300000,21.666667,0.745667,9.633333,31.766667,41.400000,26.433333,...,21.200000,78.506667,49.856667,63.793333,6.536667,8.136667,10.853333,118.936667,117.220000,100.950000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7941,232402150820,2024-02-15,2324,UTA,0,-3,4.0,44.433333,90.866667,0.489700,13.133333,36.233333,0.363533,20.433333,24.700000,0.828867,11.933333,34.633333,46.566667,29.700000,...,27.083333,76.606667,52.463333,64.526667,6.416667,8.793333,11.586667,121.050000,118.746667,99.610000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7942,232402150821,2024-02-15,2324,POR,0,-5,3.0,39.600000,88.500000,0.447767,11.633333,32.200000,0.361833,18.400000,22.966667,0.802033,11.466667,30.333333,41.800000,23.033333,...,22.140000,78.226667,51.766667,65.160000,7.886667,10.340000,13.586667,117.103333,110.983333,96.613333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8222,232403291099,2024-03-29,2324,OKC,1,-1,7.0,44.433333,90.400000,0.493233,13.733333,35.466667,0.386233,15.933333,19.666667,0.819767,9.566667,33.666667,43.233333,26.033333,...,24.773333,75.926667,52.286667,63.256667,7.190000,11.576667,13.826667,118.200000,113.796667,99.566667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8223,232403291100,2024-03-29,2324,SAS,1,2,4.0,41.633333,89.500000,0.466767,11.933333,34.200000,0.352067,15.566667,20.000000,0.773833,10.900000,33.533333,44.433333,30.300000,...,30.380000,76.173333,53.110000,60.380000,7.656667,9.646667,11.686667,117.526667,111.160000,92.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8224,232403291101,2024-03-29,2324,DEN,0,-1,8.0,43.366667,88.466667,0.490433,10.900000,30.200000,0.357967,14.966667,19.000000,0.795967,10.866667,33.633333,44.500000,29.133333,...,23.556667,76.510000,50.830000,64.483333,7.980000,11.200000,11.486667,117.306667,108.743333,95.536667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8225,232403291102,2024-03-29,2324,UTA,0,-7,1.0,42.366667,90.233333,0.469000,12.800000,36.333333,0.350167,18.266667,21.766667,0.842800,11.733333,33.400000,45.133333,27.233333,...,27.200000,76.646667,51.530000,57.760000,8.260000,8.920000,11.130000,116.660000,112.653333,99.836667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
corr_matrix = team_avgs_last_20.select_dtypes(include=[np.number]) \
  .corr() \
  .dropna(axis=0, how='all') \
  .dropna(axis=1, how='all')

plt.figure(figsize=(55, 55))
heatmap1 = sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f")

plt.savefig("../graphs/corr_processed_20.png", dpi=300, bbox_inches='tight')
plt.close()
#plt.show()

In [24]:
corr_matrix = team_avgs_last_30.select_dtypes(include=[np.number]) \
  .corr() \
  .dropna(axis=0, how='all') \
  .dropna(axis=1, how='all')

plt.figure(figsize=(55, 55))
heatmap1 = sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f")

plt.savefig("../graphs/corr_processed_30.png", dpi=300, bbox_inches='tight')
plt.close()
#plt.show()

In [25]:
corr_matrix = team_avgs_last_40.select_dtypes(include=[np.number]) \
  .corr() \
  .dropna(axis=0, how='all') \
  .dropna(axis=1, how='all')

plt.figure(figsize=(55, 55))
heatmap1 = sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f")

plt.savefig("../graphs/corr_processed_40.png", dpi=300, bbox_inches='tight')
plt.close()
#plt.show()

In [26]:
corr_matrix = team_avgs_all_season.select_dtypes(include=[np.number]) \
  .corr() \
  .dropna(axis=0, how='all') \
  .dropna(axis=1, how='all')

plt.figure(figsize=(55, 55))
heatmap1 = sns.heatmap(corr_matrix, cmap="coolwarm", annot=True, fmt=".2f")

plt.savefig("../graphs/corr_processed_all_season.png", dpi=300, bbox_inches='tight')
plt.close()
#plt.show()

In [27]:
pearson_corr_with_target = corr_matrix["home_win"].dropna()

spearman_corr_matrix = team_avgs_all_season.select_dtypes(include=[np.number]) \
    .corr(method="spearman") \
    .dropna(axis=0, how='all') \
    .dropna(axis=1, how='all')
spearman_corr_with_target = spearman_corr_matrix["home_win"].dropna()

spearman_corr_sorted = spearman_corr_with_target.sort_values()
pearson_corr_sorted = pearson_corr_with_target.loc[spearman_corr_sorted.index]

spearman_corr_sorted = spearman_corr_sorted.drop(["home_win", "game_id"])
pearson_corr_sorted = pearson_corr_sorted.drop(["home_win", "game_id"])

prc_columns = ['home_TS%', 'home_eFG%', 'home_3PAr', 'home_FTr',
       'home_ORB%', 'home_DRB%', 'home_TRB%', 'home_AST%', 'home_STL%',
       'home_BLK%', 'home_TOV%', 'home_ORtg', 'home_DRtg', 'away_TS%', 'away_eFG%',
       'away_3PAr', 'away_FTr', 'away_ORB%', 'away_DRB%', 'away_TRB%',
       'away_AST%', 'away_STL%', 'away_BLK%', 'away_TOV%', 'away_ORtg',
       'away_DRtg', 'away_TS%']
highlight_color_prc = "red"

totals_columns = ['home_FG', 'home_FGA', 'home_FG%', 'home_3P', 'home_3PA',
   'home_3P%', 'home_FT', 'home_FTA', 'home_FT%', 'home_ORB', 'home_DRB',
   'home_TRB', 'home_AST', 'home_STL', 'home_BLK', 'home_TOV', 'home_PF',
   'home_PTS',  'home_TS%', 'away_FG', 'away_FGA',
   'away_FG%', 'away_3P', 'away_3PA', 'away_3P%', 'away_FT', 'away_FTA',
   'away_FT%', 'away_ORB', 'away_DRB', 'away_TRB', 'away_AST', 'away_STL',
   'away_BLK', 'away_TOV', 'away_PF', 'away_PTS', 'home_Pace', 'away_Pace']

add_columns = [col for col in spearman_corr_sorted.index if col not in prc_columns and col not in totals_columns]

highlight_color_totals = "blue"

plt.figure(figsize=(10, 14))

bar_width = 0.4
indices = range(len(spearman_corr_sorted))

plt.barh(indices, pearson_corr_sorted.values, bar_width, label="Pearson", color="dodgerblue")
plt.barh([i + bar_width for i in indices], spearman_corr_sorted.values, bar_width, label="Spearman", color="orange")

for i, label in enumerate(spearman_corr_sorted.index):
    if label in prc_columns:
        color = "red"
    elif label in totals_columns:
        color = highlight_color_totals
    else:
        color = "black"

    plt.text(
        -0.4,
        i + bar_width / 2,
        label,
        color=color,
        va="center",
        fontsize=12,
    )

plt.title("Korelacje z 'home_win'", fontsize=16)
plt.xlabel("Współczynniki korelacji", fontsize=14)
plt.tight_layout()
plt.legend(fontsize=12)

plt.savefig("../graphs/corr_target.png", dpi=300, bbox_inches='tight')
plt.close()

plt.show()

In [28]:
with sqlite3.connect('../data/transformed/team_moving_avgs_merged.sqlite') as con:
    team_avgs_last_20.to_sql('team_last_20', con, if_exists='replace', index=False)
    team_avgs_last_30.to_sql('team_last_30', con, if_exists='replace', index=False)
    team_avgs_last_40.to_sql('team_last_40', con, if_exists='replace', index=False)
    team_avgs_all_season.to_sql('team_all_season', con, if_exists='replace', index=False)